## Lasso Regression — California Housing

In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
housing = fetch_california_housing()

In [4]:
X = pd.DataFrame(
    housing.data,
    columns = housing.feature_names
)

In [5]:
y = housing.target

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Scaling
**Fix:** `X_test` must be transformed using the scaler already fit on `X_train` (`.transform`), not re-fit on itself (`.fit_transform`). Re-fitting on the test set means the test features get scaled by the test set's own mean/std instead of the training distribution — this silently distorts test metrics.

In [7]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
model = Lasso(alpha = 0.1)

In [9]:
model.fit(X_train_scaled,y_train)

Lasso(alpha=0.1)

In [10]:
y_pred = model.predict(X_test_scaled)

### Metrics
All metrics use `(y_true, y_pred)` order consistently.

In [11]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

### Adjusted R²
**Fix:** `n` and `p` now come from `X_test`, since `r2` was computed on the test set.

In [12]:
n = X_test.shape[0]   # test set size, not full dataset
p = X_test.shape[1]

adj_r2 = 1 - ((1-r2)*(n-1))/(n-p-1)

In [13]:
print("MAE: ", round(mae,4))
print("MSE: ", round(mse,4))
print("R2: ", round(r2,4))
print("adj_r2", round(adj_r2,4))

MAE:  0.6222
MSE:  0.6796
R2:  0.4814
adj_r2 0.4804


### Cross-validated alpha search (leakage fixed)
A `Pipeline` bundles the scaler and model together so `cross_val_score` refits the scaler on each training fold only.

In [14]:
alphas = [0.001, 0.01, 0.1, 1, 10, 100]

cv = {}
for alpha in alphas:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=alpha))
    ])
    cv_scores = cross_val_score(pipe, X, y, cv=5, scoring='r2')
    cv[alpha] = float(round(cv_scores.mean(),4))

In [15]:
cv

{0.001: 0.553,
 0.01: 0.5485,
 0.1: 0.4311,
 1: -0.0892,
 10: -0.0892,
 100: -0.0892}

In [16]:
for a in alphas:
    print("\nAverage CV R\u00b2 for", a, "is: ", cv[a])


Average CV R² for 0.001 is:  0.553

Average CV R² for 0.01 is:  0.5485

Average CV R² for 0.1 is:  0.4311

Average CV R² for 1 is:  -0.0892

Average CV R² for 10 is:  -0.0892

Average CV R² for 100 is:  -0.0892


### Refit final model with the best alpha from CV

In [17]:
best_alpha = max(cv, key=cv.get)
print("Best alpha:", best_alpha)

model = Lasso(alpha=best_alpha)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

r2 = r2_score(y_test, y_pred)
print("Test R\u00b2 with best alpha:", round(r2, 4))

Best alpha: 0.001
Test R² with best alpha: 0.5769
